# 01 · Ultrasound dataset preparation
**OMARI Seminar Series — From Pixels to Predictions: Developing AI Models for Medical Image Interpretation**

Load FETAL_PLANES_DB, examine class balance and acquisition metadata, build **patient-level** splits that keep the authors' test set, detect near-duplicate images, visualise each model's preprocessing, and save reusable split manifests plus the statistics used by the website.

| | |
|---|---|
| **Runtime** | CPU is fine (≈5 min; duplicate search ≈5–10 min for 12,400 images) |
| **Needs** | Internet access to zenodo.org (or the extracted archive) |
| **Produces** | `splits/split_manifest.csv`, `splits/near_duplicates.csv`, `dataset_stats.json` |
| **Modes** | `smoke` (synthetic data, random weights: checks the code only) · `demo` (real data subset, few epochs) · `full` (complete experiment) |

> Results from `smoke` mode are meaningless by construction and must never be exported to the website.
> Every artifact written by these notebooks records the script, checkpoint hash and parameters that produced it.

In [ ]:
#@title 0 · Setup: repository, packages, data and artifact folders { display-mode: "form" }
import os, sys, subprocess, json, random, time
MODE = "demo"  #@param ["smoke", "demo", "full"]
REPO_URL = "https://github.com/YOUR-ORG/omari-workshop.git"  #@param {type:"string"}
DATA_ROOT = "/content/data/FETAL_PLANES_ZENODO"  #@param {type:"string"}
ARTIFACTS = "/content/omari_artifacts"  #@param {type:"string"}
USE_DRIVE_FOR_ARTIFACTS = False  #@param {type:"boolean"}
MODE = os.environ.get("OMARI_MODE", MODE)            # lets CI / nbclient override
ARTIFACTS = os.environ.get("OMARI_ARTIFACTS", ARTIFACTS)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE_FOR_ARTIFACTS:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/omari_artifacts"

# 1) make `omari_us` importable: installed package > local checkout (../python) > clone REPO_URL
def _find_repo():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/omari-workshop"]:
        if os.path.isdir(os.path.join(cand, "python", "omari_us")):
            return cand
    return None
REPO = _find_repo()
if REPO is None:
    if "YOUR-ORG" in REPO_URL:
        raise RuntimeError("Set REPO_URL to the repository that contains python/omari_us (or upload the repo to /content/omari-workshop).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/omari-workshop"], check=True)
    REPO = "/content/omari-workshop"
sys.path.insert(0, os.path.join(REPO, "python"))

# 2) packages (Colab already has torch/torchvision; timm and friends are added)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.12", "scikit-learn", "scipy", "pyyaml", "opencv-python-headless", "gdown"], check=True)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import omari_us
from omari_us.utils import seed_everything, get_device, CLASSES, CLASS_SHORT
SEED = 42
seed_everything(SEED)
DEVICE = get_device()
os.makedirs(ARTIFACTS, exist_ok=True)
SPLITS_CSV = os.path.join(ARTIFACTS, "splits", "split_manifest.csv")
RUNS = os.path.join(ARTIFACTS, "runs")
print(f"omari_us {omari_us.__version__} | torch {torch.__version__} | device {DEVICE} | MODE={MODE} | artifacts → {ARTIFACTS}")
if DEVICE.type == "cpu" and MODE != "smoke":
    print("⚠ No GPU. In Colab: Runtime → Change runtime type → T4 GPU. CPU training of these models is impractically slow.")

In [ ]:
#@title 1 · Data: download FETAL_PLANES_DB (2.1 GB, CC BY 4.0) or build the synthetic smoke dataset { display-mode: "form" }
from omari_us.datasets import read_metadata, make_splits, synthetic_metadata, save_split_manifest, load_split_manifest, ZENODO_URL, ZENODO_MD5

def ensure_fetal_planes(root=DATA_ROOT):
    """Download + verify + extract the Zenodo archive if the metadata CSV is not already present."""
    import glob, hashlib
    if glob.glob(os.path.join(root, "**", "FETAL_PLANES_DB_data.csv"), recursive=True):
        return
    os.makedirs(root, exist_ok=True)
    z = os.path.join(os.path.dirname(root), "FETAL_PLANES_ZENODO.zip")
    if not os.path.exists(z):
        print("downloading", ZENODO_URL)
        subprocess.run(["curl", "-L", "--fail", "-o", z, ZENODO_URL], check=True)
    md5 = hashlib.md5(open(z, "rb").read()).hexdigest()
    assert md5 == ZENODO_MD5, f"md5 mismatch: {md5}"
    subprocess.run(["unzip", "-q", "-n", z, "-d", root], check=True)

if MODE == "smoke":
    SMOKE_DIR = os.path.join(ARTIFACTS, "smoke_data")
    meta = synthetic_metadata(60, (2, 5), seed=0, image_dir=os.path.join(SMOKE_DIR, "Images"), size=(200, 150))
    print("SMOKE MODE: synthetic grey images; every number below is meaningless.")
else:
    ensure_fetal_planes()
    meta = read_metadata(DATA_ROOT)
print(meta.shape)
meta.head()

## 1 · Metadata and class balance
The CSV has one row per image: `Image_name; Patient_num; Plane; Brain_plane; Operator; US_Machine; Train`. `Train` holds the authors' split
(patients ordered by first visit; first half → train). Column names carry trailing spaces in the release, which `read_metadata` strips.

In [ ]:
counts = pd.crosstab(meta["Plane"], meta["Train"].map({1: "train", 0: "test"}))
counts["total"] = counts.sum(1)
display(counts.sort_values("total", ascending=False))
ax = counts["total"].sort_values().plot.barh(figsize=(6, 3), color="#3d4b55"); ax.set_xlabel("images"); plt.show()
print(f"{meta.Patient_num.nunique()} patients, {len(meta)} images, {len(meta)/meta.Patient_num.nunique():.2f} images/patient")

## 2 · Representative images (two per class)

In [ ]:
from PIL import Image
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for c in range(6):
    rows = meta[meta.label == c].sample(2, random_state=SEED)
    for r, (_, row) in enumerate(rows.iterrows()):
        im = Image.open(row.path).convert("L"); axes[r, c].imshow(im, cmap="gray"); axes[r, c].axis("off")
        axes[r, c].set_title(f"{CLASS_SHORT[c]}\n{im.size[0]}×{im.size[1]}", fontsize=9)
plt.tight_layout(); plt.show()

## 3 · Acquisition metadata: machines, operators, image sizes
Acquisition variables that correlate with class are candidate shortcuts (Chapter 6) and sources of domain shift (Chapter 4).

In [ ]:
if "US_Machine" in meta: display(pd.crosstab(meta["Plane"], meta["US_Machine"], normalize="columns").round(2))
if "Operator" in meta: display(pd.crosstab(meta["Plane"], meta["Operator"]).T)
sample = meta.sample(min(800, len(meta)), random_state=SEED)
sizes = pd.Series([Image.open(p).size for p in sample.path], index=sample.index)
print("most common image sizes (sample):"); print(sizes.value_counts().head(8))

## 4 · Patient-level splits
We keep the authors' test split and carve a **patient-disjoint, class-stratified** validation set (10%) from the training half
with `StratifiedGroupKFold`. `check_patient_disjoint` raises if any patient appears in two splits.

In [ ]:
splits = make_splits(meta, val_fraction=0.15 if MODE == "smoke" else 0.1, seed=SEED)
from omari_us.datasets import check_patient_disjoint
print("patients per split:", check_patient_disjoint(splits))
display(pd.crosstab(splits["Plane"], splits["split"]))

### What a random image-level split would have done
The share of test images whose patient also has images in training, measured empirically.

In [ ]:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(meta, test_size=0.4, random_state=SEED, stratify=meta.label)
leak = te.Patient_num.isin(set(tr.Patient_num)).mean()
print(f"random image-level split: {leak:.1%} of test images share a patient with training (patient-level split: 0%)")

## 5 · Near-duplicate detection
64-bit difference hashes; pairs within `MAX_HAMMING` bits are flagged. Inspect a few pairs before drawing conclusions: dHash also matches different frames of very similar views.

In [ ]:
from omari_us.datasets import find_near_duplicates
MAX_HAMMING = 4  #@param {type:"integer"}
dup_df = splits if MODE == "full" else splits.sample(min(3000, len(splits)), random_state=SEED).reset_index(drop=True)
dups = find_near_duplicates(dup_df, MAX_HAMMING)
print(f"{len(dups)} near-duplicate pairs in {len(dup_df)} images; same patient: {dups.same_patient.mean() if len(dups) else 0:.1%}")
if len(dups):
    cross = dups[(dups.split_a == "test") ^ (dups.split_b == "test")]
    print(f"pairs crossing the train/val ↔ test boundary: {len(cross)}")
    fig, axes = plt.subplots(2, min(5, len(dups)), figsize=(14, 5), squeeze=False)
    for k, (_, r) in enumerate(dups.head(axes.shape[1]).iterrows()):
        for j, idx in enumerate((r.a, r.b)):
            axes[j, k].imshow(Image.open(dup_df.path.iloc[idx]).convert("L"), cmap="gray"); axes[j, k].axis("off")
            axes[j, k].set_title(dup_df.Image_name.iloc[idx][:22], fontsize=7)
    plt.show()

## 6 · Preprocessing and augmentation, per model
Each model keeps the preprocessing its pretrained weights expect (see `omari_us/datasets/transforms.py`).

In [ ]:
from omari_us.datasets import input_spec, build_transforms, denormalize
row = splits[splits.split == "train"].iloc[0]
im = Image.open(row.path).convert("L")
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for i, mid in enumerate(["densenet161", "vit_b16", "usf_mae"]):
    spec = input_spec(mid); ev, tr = build_transforms(spec, False), build_transforms(spec, True)
    axes[i, 0].imshow(denormalize(ev(im), spec), cmap="gray"); axes[i, 0].set_title(f"{mid} eval\n{spec.interpolation}, μ={spec.mean[0]}", fontsize=8)
    for j in range(1, 5):
        axes[i, j].imshow(denormalize(tr(im), spec), cmap="gray"); axes[i, j].set_title("train aug", fontsize=8)
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.show()

## 7 · Save manifests and website statistics

In [ ]:
from omari_us.datasets import dataset_stats
from omari_us.utils import write_json
save_split_manifest(splits, SPLITS_CSV)
stats = dataset_stats(splits, dups if MODE == "full" else None, MAX_HAMMING, script="notebooks/01_dataset_preparation.ipynb")
write_json(stats, os.path.join(ARTIFACTS, "dataset_stats.json"), indent=1)
if len(dups): dups.to_csv(os.path.join(ARTIFACTS, "splits", "near_duplicates.csv"), index=False)
print("wrote", SPLITS_CSV, "and dataset_stats.json")
print("Copy dataset_stats.json to website/public/experiments/ (not in smoke mode) to replace the literature counts on the website.")
assert stats["split"]["patient_overlap"] == 0

## Expected outputs (full mode)
- 12,400 images from 1,792 patients; six classes with "Other" the largest and the abdomen the smallest class.
- `patient_overlap = 0` between the authors' train and test halves; validation ≈10% of training patients.
- A non-trivial number of near-duplicate pairs, almost all within the same patient (saved frame sequences).

## References
- Burgos-Artizzu X.P. et al. *Evaluation of deep convolutional neural networks for automatic classification of common maternal fetal ultrasound planes.* Sci. Rep. 10, 10200 (2020). Dataset: Zenodo 3904280, CC BY 4.0.
- Huang G. et al. *Densely connected convolutional networks.* CVPR 2017.
- Dosovitskiy A. et al. *An image is worth 16×16 words.* ICLR 2021; Steiner A. et al. *How to train your ViT?* TMLR 2022.
- He K. et al. *Masked autoencoders are scalable vision learners.* CVPR 2022.
- Megahed Y. et al. *USF-MAE: Ultrasound self-supervised foundation model with masked autoencoding.* Biomed. Signal Process. Control 122, 110313 (2026). https://github.com/Yusufii9/USF-MAE
- Zhou B. et al. CAM, CVPR 2016 · Selvaraju R.R. et al. Grad-CAM, ICCV 2017 · Chattopadhay A. et al. Grad-CAM++, WACV 2018 · Abnar & Zuidema, attention rollout, ACL 2020 · Chefer H. et al., CVPR 2021 · Adebayo J. et al., sanity checks, NeurIPS 2018 · Petsiuk V. et al., RISE / deletion–insertion, BMVC 2018 · Guo C. et al., calibration, ICML 2017.